### Writing to sentinel-5P zarr store

Necessary imports

In [ ]:
import zarr 

For now we will only write CO data to the zarr store. For this we will go through the folder containing our netCDF files and reproject them to EUQI7 grid, as some files don't extent over Austria a potential error is overriden.

In [ ]:
datasets = []
for file in os.listdir(f"CO/"):
        path = os.path.join(f"CO/", file)
        if file.startswith("S5P_OFFL"):
            try:
                datasets.append(reproject_data(path, qa_value, is_no2))
            except (ValueError) as e:
                print(f"Warning file {file}: {e}")
                continue
        else:
            continue

The datasets in the datasets list are now concatenated, if the timecoordinate is the same for two datasets the data is averaged.

In [ ]:
 merged = merge_mean_by_time(datasets)

Now we can open the zarr store

In [ ]:
store = zarr.storage.LocalStore(store_path)
group = zarr.group(store=store, path=product_type)

The time indexes are then calculated

In [ ]:
time_origin = np.datetime64("2018-04-01")
time_min = (merged.time.min().values.astype("datetime64[D]") - time_origin).astype("int64")
time_max = (merged.time.max().values.astype("datetime64[D]") - time_origin).astype("int64")

Next the data is written to the correct position in the zarr store. The correct nodata value and scale factor is applied before writing

In [ ]:
for var in merged.data_vars:
    scale = group[var].attrs["scale_factor"]
    fill_value = group[var].attrs["_FillValue"]
    data = np.nan_to_num(np.round(group[var].values/scale), nan=fill_value)
    group[var][time_min:time_max, :, :] = data